# Sesión 1 · Línea base: ¿firmarías esto?

**Objetivo.** Ver qué devuelve un agente cuando le damos la pregunta tal cual, sin instrucciones
sobre cómo analizarla, y decidir si pondrías tu nombre en ese resultado.

**Al terminar** vas a tener cinco corridas guardadas en `resultados/s1.json` y la primera versión
de tu tarjeta de confianza. **Tiempo:** 80 minutos.

In [ ]:
# --- Configuración: correr una vez al empezar ---
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
import matplotlib.pyplot as plt
from agente.llm import desde_entorno
from agente.harness import ejecutar_agente, correr_varias, leer_traza, con_especificacion, SISTEMA_BASE
from agente.evaluacion import tarjeta, huerfanos, guardar, cargar, cargar_respuesta, diagnosticos_ejecutados

# os.environ["LLM_PROVEEDOR"] = "ollama"   # o "anthropic" / "openai"; si no lo configurás, se usa el simulador
# os.environ["LLM_MODELO"] = "qwen3.5:4b"   # el modelo que descargaste (ver INSTALACION.md)
llm = desde_entorno()
df = pd.read_csv(RAIZ / "datos" / "programa_x.csv")
TRAZAS = RAIZ / "trazas" / "s1"
RES = RAIZ / "resultados"; RES.mkdir(exist_ok=True)
VERDAD, TOLERANCIA = cargar_respuesta(RAIZ)
PREGUNTA = "¿Cuánto aumentó el programa el ingreso posterior de quienes participaron?"

## 1. Los datos (10 min)

Leé `datos/diccionario.md`. Después mirá los datos vos, antes que el agente.

In [ ]:
print(df.shape)
df.head()

In [ ]:
df.groupby("participa")[["edad", "anios_educ", "ingreso_t1", "ingreso_t2", "ingreso_post"]].mean().round(0)

**Antes de seguir, anotá tu predicción.** ¿El programa aumentó o redujo los ingresos? ¿En cuánto,
más o menos? ¿Qué te preocupa de comparar participantes con no participantes en esta tabla?

In [ ]:
MI_PREDICCION = None  # completá con un número en pesos del programa

## 2. El agente por dentro (15 min)

Abrí `agente/harness.py`. Tiene tres piezas:

1. **Un modelo** (`llm.completar`) que recibe la conversación y devuelve texto.
2. **Una herramienta**: si el texto trae un bloque de Python, lo ejecutamos y le devolvemos lo que imprimió.
3. **Un ciclo** que se repite hasta que el modelo devuelve un JSON con la respuesta final.

Nada más. Todo lo que hagamos en el seminario para volverlo confiable se agrega alrededor de este ciclo.
Este es el único mensaje de sistema que recibe hoy:

In [ ]:
print(SISTEMA_BASE)

## 3. Una corrida (10 min)

In [ ]:
c = ejecutar_agente(PREGUNTA, df, llm, carpeta_trazas=TRAZAS, etiqueta="s1")
print(json.dumps(c.respuesta, ensure_ascii=False, indent=1))

Cada paso queda registrado en `trazas/s1/`. ¿Qué código corrió? ¿Qué vio el modelo antes de responder?

In [ ]:
for e in leer_traza(c.traza):
    if e["tipo"] == "codigo":
        print(f"--- paso {e['paso']}: código ---\n{e['codigo']}\n--- salida ---\n{e['salida'][:800]}\n")

## 4. Cinco corridas (20 min)

Misma pregunta, mismo modelo, cinco veces.

In [ ]:
res = correr_varias(5, PREGUNTA, df, llm, carpeta_trazas=TRAZAS, etiqueta="s1")
guardar(res, RES / "s1.json")
res[["metodo", "estimacion", "ic_inferior", "ic_superior", "abstiene"]]

In [ ]:
res.afirmaciones.explode().dropna().tolist()

## 5. ¿Firmarías? (15 min)

En equipo, respondan por escrito:

1. ¿Las cinco corridas cuentan la misma historia?
2. ¿Qué método eligió el agente? ¿Explicó por qué?
3. ¿Qué diagnósticos corrió? ¿Cuáles debería haber corrido?
4. ¿Hay algún número en las afirmaciones que no veas en ninguna salida?
5. ¿Le pondrías tu firma a alguna de estas respuestas? ¿A cuál, y por qué?

## 6. Tarjeta de confianza, versión 0

Sin la respuesta correcta solo podemos llenar parte de la tarjeta.

In [ ]:
tarjeta(res, VERDAD, TOLERANCIA)

## 7. La revelación (10 min)

Este programa tuvo, además, un **experimento**: un grupo de postulantes fue sorteado para no participar.
Compararse con ese grupo da el efecto verdadero. El docente lo revela ahora y entrega `respuesta.json`:
copialo a la raíz del kit y corré la celda.

In [ ]:
VERDAD, TOLERANCIA = cargar_respuesta(RAIZ)
print(f"efecto verdadero: {VERDAD} · tolerancia: ± {TOLERANCIA}")
tarjeta(res, VERDAD, TOLERANCIA)

## Para la próxima

- **Leer:** Huyen, *AI Engineering*, cap. 3–4; Kapoor et al. (2025), «AI Agents That Matter».
- **Pensar:** si tuvieras que escribirle al agente cómo analizar esto, ¿qué le dirías? Eso es la sesión 2.